# **DEEP LEARNING MODELLING: CNN-LSTM USING GEOGRAPHICAL COORDINATES**

## **1. LIBRARIES**

### **1.1 DEPENDENCIES**

In [ ]:
! pip install adlfs azure-storage-blob
! pip install tensorflow
! pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.9/187.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 11.3 MB/s eta 0:00:00


### **1.2 LIBRARIES**

In [ ]:
# STANDARD LIBRARY
import os
import io
import random

# DATA MANIPULATION AND ANALYSIS
import pandas as pd
import numpy as np

# DATA VISUALISATION
import matplotlib.pyplot as plt
import seaborn as sns

# AZURE CLOUD STORAGE
from adlfs import AzureBlobFileSystem
from azure.storage.blob import BlobServiceClient

# REGULAR EXPRESSIONS
import re

# TIME SERIES
from sklearn.model_selection import TimeSeriesSplit

# STANDARIZATION
from sklearn.preprocessing import StandardScaler

# EVALUATION METRICS
from sklearn.metrics import r2_score

# PROGRESS BAR
from tqdm import tqdm

# TENSORFLOW
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Conv1D, BatchNormalization, Dropout, LSTM, Dense, Input
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.losses import Huber

In [ ]:
# SEEDS
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
print(tf.__version__)


2.19.0


In [ ]:
# Correctly set pandas display option for max columns and rows
pd.options.display.max_columns = None
pd.options.display.max_rows = None

## **2. LOADING DATASETS FROM AZURE BLOB STORAGE**

In [ ]:
def load_csv_from_blob(blob_name):
    blob_client = container_client.get_blob_client(blob_name)
    stream = blob_client.download_blob()
    data_bytes = stream.readall()
    data_str = data_bytes.decode("utf-8")
    return pd.read_csv(io.StringIO(data_str))


# Connection configuration
AZURE_STORAGE_ACCOUNT = "researchprojectx24104515"
AZURE_STORAGE_KEY = "bxpexO6i+Hz6n1WiipTn+sTCuLPGMS1BogMERrIrHd16DpQ0GLfQ0R33yrSw4MxsDomq5yNMgw1o+AStlx/MjA=="
connection_string = f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

fs = AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

In [ ]:
# Establish connection to cycle
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "preprocessed.csv"
data = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {data.shape}")

Loaded 'preprocessed.csv' with shape: (53424, 44)


In [ ]:
print("Preprocessed Data:")
print(data.shape)
data.head()

Preprocessed Data:
(53424, 44)


,date and time,CYC_Clontarf James Larkin Rd,CYC_Clontarf Pebble Beach Carpark,CYC_Grove Road Totem,CYC_North Strand Rd N B,CYC_Richmond Street Cyclists 1 Merged,CYC_Richmond Street Cyclists 2 Merged,FTF_Aston Quay Merged,FTF_Capel St Mary St Merged,FTF_College St Dame St Merged,FTF_Dame St Merged,FTF_Dolier St Burgh Quay Merged,FTF_Grafton St Monsoon Merged,FTF_Grafton St Nassau St Suffolk St,FTF_Henry St Merged,FTF_Mary St Merged,FTF_Oconnell St Princes St North Merged,FTF_Talbot St Guineys Merged,FTF_Westmoreland St East Fleet St Merged,FTF_Westmoreland St West Merged,WTH_rain,WTH_temp,WTH_wetb,WTH_dewpt,WTH_vappr,WTH_rhum,WTH_msl,TEMP_year,TEMP_holiday,WTH_temp_3h,WTH_rain_int,WTH_dew_spread,WTH_prob_cond,WTH_severe,TEMP_hour_sin,TEMP_hour_cos,TEMP_day_sin,TEMP_day_cos,TEMP_month_sin,TEMP_month_cos,TEMP_weekend,TEMP_mor_peak,TEMP_eve_peak,TEMP_bus_hours
0,2019-01-01 00:00:00,0.0,0.0,12.0,16.0,0.0,0.0,0.0,238.0,0.0,0.0,0.0,140.0,0.0,597.0,163.0,1914.0,0.0,1670.0,1988.0,0.0,9.6,7.8,5.6,9.1,76.0,1035.1,2019,0,0.000000,0,4.0,0,0,0.000000,1.000000,0.781831,0.62349,0.5,0.866025,0,0,0,0
1,2019-01-01 01:00:00,0.0,0.0,17.0,14.0,0.0,0.0,0.0,173.0,0.0,0.0,0.0,215.0,0.0,359.0,102.0,885.0,0.0,767.0,1270.0,0.0,8.6,7.1,5.3,8.9,79.0,1035.1,2019,0,0.000000,0,3.3,0,0,0.258819,0.965926,0.781831,0.62349,0.5,0.866025,0,0,0,0
2,2019-01-01 02:00:00,0.0,0.0,18.0,9.0,0.0,0.0,0.0,121.0,0.0,0.0,0.0,210.0,0.0,317.0,63.0,984.0,0.0,642.0,1589.0,0.0,8.3,6.9,5.3,8.9,81.0,1034.9,2019,0,8.833333,0,3.0,0,0,0.500000,0.866025,0.781831,0.62349,0.5,0.866025,0,0,0,0
3,2019-01-01 03:00:00,0.0,0.0,18.0,15.0,0.0,0.0,0.0,174.0,0.0,0.0,0.0,204.0,0.0,313.0,59.0,935.0,0.0,582.0,1534.0,0.0,9.1,7.6,5.8,9.2,79.0,1035.5,2019,0,8.666667,0,3.3,0,0,0.707107,0.707107,0.781831,0.62349,0.5,0.866025,0,0,0,0
4,2019-01-01 04:00:00,0.0,0.0,12.0,8.0,0.0,0.0,0.0,82.0,0.0,0.0,0.0,88.0,0.0,172.0,46.0,390.0,0.0,143.0,610.0,0.0,9.2,7.7,5.9,9.3,79.0,1035.7,2019,0,8.866667,0,3.3,0,0,0.866025,0.500000,0.781831,0.62349,0.5,0.866025,0,0,0,0


In [ ]:
# Establish connection to counters dataset
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "counters.csv"
counters = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {counters.shape}")

Loaded 'counters.csv' with shape: (31, 3)


In [ ]:
print("Counter Locations Data:")
print(counters.shape)
counters

Counter Locations Data:
(31, 3)


,Target,Latitude,Longitude
0,CYC_Clontarf James Larkin Rd,53.368540,-6.170320
1,CYC_Clontarf Pebble Beach Carpark,53.358250,-6.191070
2,CYC_Grove Road Totem,53.329644,-6.270178
3,CYC_North Strand Rd N B,53.358520,-6.241270
4,CYC_Richmond Street Cyclists 1 Merged,53.332160,-6.264400
5,CYC_Richmond Street Cyclists 2 Merged,53.332160,-6.264400
6,FTF_Aston Quay Merged,53.346680,-6.259880
7,FTF_Bachelors Walk Merged,53.347440,-6.260100
8,FTF_Baggot St Lower Merged,53.334540,-6.245620
9,FTF_Baggot St Upper Mespil Rd Bank,53.333770,-6.244510


### **2.1 DEFINE FEATURE CATEGORIES**

In [ ]:
# Interaction features created with weather features
interactionf = ["WTH_dew_spread", "WTH_prob_cond", "WTH_severe"]

# Using regex to get all temporal and weather prefix patterns
temp_pattern = re.compile(r"^TEMP_")
wth_patterb = re.compile(r"^WTH_")

# Using regex to get all temporal and weather features
temporalf = [col for col in data.columns if temp_pattern.match(col)]
weatherf = [col for col in data.columns if wth_patterb.match(col) and col not in interactionf]

In [ ]:
weatherf

['WTH_rain',
 'WTH_temp',
 'WTH_wetb',
 'WTH_dewpt',
 'WTH_vappr',
 'WTH_rhum',
 'WTH_msl',
 'WTH_temp_3h',
 'WTH_rain_int']

In [ ]:
temporalf

['TEMP_year',
 'TEMP_holiday',
 'TEMP_hour_sin',
 'TEMP_hour_cos',
 'TEMP_day_sin',
 'TEMP_day_cos',
 'TEMP_month_sin',
 'TEMP_month_cos',
 'TEMP_weekend',
 'TEMP_mor_peak',
 'TEMP_eve_peak',
 'TEMP_bus_hours']

In [ ]:
# Combine all features
fcolumns = weatherf + temporalf + interactionf

# Extract cycle and footfall targets
cycle_t = [col for col in data.columns if col.startswith("CYC_")]
footfall_t = [col for col in data.columns if col.startswith("FTF_")]
targets = cycle_t + footfall_t

# Prepare feature matrix
X = data[fcolumns].copy()
y = data[targets].copy()

# Handle missing values
X = X.ffill().bfill()
y = y.fillna(0)

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")
print(f"Feature columns: {len(fcolumns)}")

Features shape: (53424, 24)
Targets shape: (53424, 19)
Feature columns: 24


In [ ]:
targets

['CYC_Clontarf James Larkin Rd',
 'CYC_Clontarf Pebble Beach Carpark',
 'CYC_Grove Road Totem',
 'CYC_North Strand Rd N B',
 'CYC_Richmond Street Cyclists 1 Merged',
 'CYC_Richmond Street Cyclists 2 Merged',
 'FTF_Aston Quay Merged',
 'FTF_Capel St Mary St Merged',
 'FTF_College St Dame St Merged',
 'FTF_Dame St Merged',
 'FTF_Dolier St Burgh Quay Merged',
 'FTF_Grafton St Monsoon Merged',
 'FTF_Grafton St Nassau St Suffolk St',
 'FTF_Henry St Merged',
 'FTF_Mary St Merged',
 'FTF_Oconnell St Princes St North Merged',
 'FTF_Talbot St Guineys Merged',
 'FTF_Westmoreland St East Fleet St Merged',
 'FTF_Westmoreland St West Merged']

## **3. CNN / LSTM**

### **3.1 DEFINE CNN-LSTM SEQUENCES AND TIME SERIES SPLIT**

In [ ]:
# Create CNN-LSTM sequences

def cnnlstm_seqs(data, targets, features, counters, seq_len=24):

    # Scale all features
    f_scaler = MinMaxScaler()
    features = f_scaler.fit_transform(data[features])

    # Containers for results
    X_data = {}
    y_data = {}
    t_scalers = {}

    # Processing separately each target
    for t in targets:
        # Coordinates from the target
        lat = counters.loc[counters["Target"] == t, "Latitude"].values[0]
        lon = counters.loc[counters["Target"] == t, "Longitude"].values[0]

        # Extract target column and reshape to 2D
        y = data[t].values.reshape(-1, 1)

        # Fitting a scaler
        t_scaler = MinMaxScaler()
        y_scaled = t_scaler.fit_transform(y)
        X, y_seq = [], []

        # Create sliding window sequences
        for i in range(seq_len, len(features)):
            seq = features[i-seq_len:i]

            # Create a sequence lenght × 2 array with repeated coordinates
            coords = np.tile([lat, lon], (seq_len, 1))

            #Concatenate sequences with coordinates
            seq_with_coords = np.concatenate([seq, coords], axis=1)
            X.append(seq_with_coords)
            y_seq.append(y_scaled[i])

        # Converting data to NumPy arrays to store in dictionaries
        X_data[t] = np.array(X, dtype=np.float32)
        y_data[t] = np.array(y_seq, dtype=np.float32)
        t_scalers[t] = t_scaler

    return X_data, y_data, f_scaler, t_scalers


# Time Series Split
def ts_split(X, y, train_ratio=0.7, val_ratio=0.15):

    # Total number of sequences
    n_samples = len(X)

    # Index boundaeries calculated by ratios
    train_end = int(n_samples * train_ratio)
    val_end = int(n_samples * (train_ratio + val_ratio))

    # Slice sets sequentially as a Time Series
    X_train = X[:train_end]
    y_train = y[:train_end]
    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]
    X_test = X[val_end:]
    y_test = y[val_end:]

    return X_train, X_val, X_test, y_train, y_val, y_test

### **3.2 BUILD CNN-LSTM MODEL**

In [ ]:
def build_cnnlstm(input_shape, conv_filters=64, lstm_units=64, dense_units=64, dropout_rate=0.2):
    # Level 2 regularisation
    reg = 1e-4

    # Batch Normalization for Training stabilisation and Dropout to avoid overfitting

    # Model
    model = Sequential()
    model.add(Input(shape=input_shape))

    # FEATURE EXTRACTION
    # First convolutional layer for feature extraction from short-term sequential data
    model.add(Conv1D(conv_filters, 3, activation="relu", padding="same",
                     kernel_regularizer=regularizers.l2(reg)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # Second convolutional layer for more abstract features and long-term temporal patterns
    model.add(Conv1D(conv_filters//2, 5, activation="relu", padding="same",
                     kernel_regularizer=regularizers.l2(reg)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # LSTM FOR LONG TERM DEPENDENCIES
    # First LSTM layer, returns to a sequence that keeps temporal information
    model.add(LSTM(lstm_units, return_sequences=True, dropout=dropout_rate, recurrent_dropout=dropout_rate,
                   kernel_regularizer=regularizers.l2(reg)))
    model.add(Dropout(dropout_rate))

    # Second LSTM layer, returns to a single output
    model.add(LSTM(lstm_units//2, return_sequences=False, dropout=dropout_rate, recurrent_dropout=dropout_rate,
                   kernel_regularizer=regularizers.l2(reg)))
    model.add(Dropout(dropout_rate))

    # Dense layer for learned features before final prediction
    model.add(Dense(dense_units, activation="relu", kernel_regularizer=regularizers.l2(reg)))
    model.add(Dropout(dropout_rate))

    # Single output neuron with no negative predictions
    model.add(Dense(1, activation="relu"))

    # Model compilation
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=Huber(delta=1.0),
        metrics=["mae"]
    )
    return model

### **3.3 TRAIN MODEL**

In [ ]:
# TRAIN MODEL
def train_cnnlstm(target_name, X, y, callbacks=None):

    # Training starts
    print(f"\nTraining model for: {target_name}")

    # Split the sequences into train, test and validation sets
    X_train, X_val, X_test, y_train, y_val, y_test = ts_split(X, y)

    # CNN-LSTM model built with an input shape that matches the sequences
    model = build_cnnlstm(input_shape=X_train.shape[1:])

    # If not callbacks provided, use this as default
    if callbacks is None:
        # Stop if validation loss stops improving for 10 epochs
        early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        # Reduce learning rate if validation loss doe not improve after 5 epochs
        reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
        callbacks = [early_stopping, reduce_lr]

    # Train the model
    history = model.fit(
        X_train, y_train,
        # Maximum training Epochs
        epochs=100,
        # Large batch size for training efficiency
        batch_size=1024,
        # Monitor performance on validation set
        validation_data=(X_val, y_val),
        # Apply early stop or reduce learning rate
        callbacks=callbacks,
        # Shows training process
        verbose=1
    )

    print(f"------Training completed for: {target_name}")
    return model, history, X_test, y_test

### **3.4 EVALUTAION**

In [ ]:
def evaluate_mtarget(model, X_test, y_test, target_scaler, target_name, target_idx, n_targets):
    print(f"\n----- Evaluating model for: {target_name} ({target_idx + 1}/{n_targets})")

    # Predictions
    y_pred = model.predict(X_test)

    # Inverse transfomation of y_pred and predictions
    y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1))
    y_pred = target_scaler.inverse_transform(y_pred)

    # Round to whole numbers
    y_pred = np.round(y_pred).astype(int)
    y_true = np.round(y_true).astype(int)

    # Evaluation metrics
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if np.any(mask) else np.nan
    r2 = r2_score(y_true, y_pred)

    # Evalation metrics print
    print("Evaluation Metrics:")
    print(f" -RMSE : {rmse:.2f}")
    print(f" -MAE  : {mae:.2f}")
    print(f" -MAPE : {mape:.2f}%")
    print(f" -R²   : {r2:.4f}")

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }

### **3.5 APPLYING CNN-LSTM**

In [ ]:
# APPLY CNN LSTM

# Number of time steps per sequence
sequence_length = 24

# Prepare CNN-LSTM sequences (features and coordinates for each timestep)
X_cnn_lstm_dict, y_cnn_lstm_dict, feature_scaler_lstm, target_scaler_dict = cnnlstm_seqs(
    # Combines features (X) with targets (Y) into a DataFrames, including coordinates
    pd.concat([X, y], axis=1),
    targets,
    fcolumns,
    counters,
    sequence_length
)

# Shape of sequence
print("Shape of a sequence with coordinates:", X_cnn_lstm_dict[targets[0]].shape)

# Dictionaries to store results for all targets
models = {}
histories = {}
results = {}
test_sets = {}

# Total number of target variables
n_targets = len(targets)

# Train and evaluate model for each target
for idx, target_name in enumerate(targets, 1):
    print(f"\n--- Processing target {idx} out of {n_targets}: {target_name}")

    # Retrieve the prepared sequences for this target
    X_data = X_cnn_lstm_dict[target_name]
    y_data = y_cnn_lstm_dict[target_name]

    # Train the model with early stopping
    model, history, X_test, y_test = train_cnnlstm(
        target_name,
        X_data,
        y_data,
        callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
    )

    # Store the models and training history
    models[target_name] = model
    histories[target_name] = history
    test_sets[target_name] = (X_test, y_test)
    target_scaler = target_scaler_dict[target_name]

    # Evaluate the performance of the model on test set
    metrics = evaluate_mtarget(
        model,
        X_test,
        y_test,
        target_scaler,
        target_name,
        idx,
        n_targets
    )

    # Save the metrics for the evaluated target
    results[target_name] = metrics

print("\nTraining and evaluation completed for all targets.")

Shape of a sequence with coordinates: (53400, 24, 26)

--- Processing target 1 out of 19: CYC_Clontarf James Larkin Rd

Training model for: CYC_Clontarf James Larkin Rd
Epoch 1/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 19s 158ms/step - loss: 0.0286 - mae: 0.0501 - val_loss: 0.0234 - val_mae: 0.0750
Epoch 2/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - loss: 0.0177 - mae: 0.0342 - val_loss: 0.0171 - val_mae: 0.0750
Epoch 3/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 122ms/step - loss: 0.0114 - mae: 0.0291 - val_loss: 0.0133 - val_mae: 0.0750
Epoch 4/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 119ms/step - loss: 0.0077 - mae: 0.0252 - val_loss: 0.0110 - val_mae: 0.0750
Epoch 5/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 123ms/step - loss: 0.0054 - mae: 0.0215 - val_loss: 0.0096 - val_mae: 0.0750
Epoch 6/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 123ms/step - loss: 0.0040 - mae: 0.0200 - val_loss: 0.0056 - val_mae: 0.0645
Epoch 7/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 120ms/step - loss: 0.0032 - mae: 0.0199 - val_loss: 0.0082 - val_mae: 0.07

## **3.6 MODEL SUMMARY**

In [ ]:
for target_name, model in models.items():
    print(f"\n---Summary for target: {target_name}")
    model.summary()


---Summary for target: CYC_Clontarf James Larkin Rd


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 24, 64)         │         5,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 24, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 24, 32)         │        10,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165,033 (644.67 KB)

 Trainable params: 54,945 (214.63 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_36 (Conv1D)              │ (None, 24, 64)         │         5,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_36          │ (None, 24, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_90 (Dropout)            │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_37 (Conv1D)              │ (None, 24, 32)         │        10,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_37          │ (None, 24, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_91 (Dropout)            │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_36 (LSTM)                  │ (None, 24, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_92 (Dropout)            │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_37 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_93 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_94 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165,033 (644.67 KB)

 Trainable params: 54,945 (214.63 KB)

 Non-trainable params: 192 (768.00 B)

 Optimizer params: 109,896 (429.29 KB)

## **3.7 RESULTS**

In [ ]:
results = pd.DataFrame.from_dict(results, orient="index")
results.index.name = "Target"
results = results.reset_index()
print("\nSummary of results CNN-LSTM tunned with coordinates:")
results


Summary of results CNN-LSTM tunned with coordinates:


,Target,RMSE,MAE,MAPE,R2
0,CYC_Clontarf James Larkin Rd,55.647083,50.286517,1126.305334,-1.093572
1,CYC_Clontarf Pebble Beach Carpark,29.964900,18.657303,64.676267,0.701203
2,CYC_Grove Road Totem,84.338870,46.045318,77.697737,0.502168
3,CYC_North Strand Rd N B,0.000000,0.000000,NaN,1.000000
4,CYC_Richmond Street Cyclists 1 Merged,40.114191,24.192260,109.876660,0.244941
5,CYC_Richmond Street Cyclists 2 Merged,36.243203,25.135830,247.594527,0.274258
6,FTF_Aston Quay Merged,1185.883175,950.874782,226.117283,0.090613
7,FTF_Capel St Mary St Merged,677.681752,576.506866,305.739785,-0.297645
8,FTF_College St Dame St Merged,241.366822,183.045693,186.603737,0.673437
9,FTF_Dame St Merged,538.112827,453.009363,481.329281,-1.837651


### **3.8 PREDICTIONS**

In [ ]:
# Dictionary with DataFrames for all targets
predictions = {}

for target_name in targets:
    print(f"\nPredicting values for: {target_name}")
    X_test, y_test = test_sets[target_name]
    target_scaler = target_scaler_dict[target_name]

    # Scale and descale predictions
    y_pred_scaled = models[target_name].predict(X_test)
    y_pred = target_scaler.inverse_transform(y_pred_scaled)
    y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    # Index of start with date and time
    # Sequence length in LSTM sequences
    seq_len = sequence_length
    n_total = len(y_cnn_lstm_dict[target_name])
    n_train = int(n_total * 0.7)
    n_val = int(n_total * 0.15)
    start_test_idx = seq_len + n_train + n_val

    datest = data["date and time"].iloc[start_test_idx:].reset_index(drop=True)

    # DataFrame with only whole numbers and avoiding negative values
    predsDataFrame = pd.DataFrame({
        "date and time": datest,
        "Real": np.maximum(np.round(y_true).astype(int).reshape(-1), 0),
        "CNNLSTM_coords": np.maximum(np.round(y_pred).astype(int).reshape(-1), 0)
    })

    predictions[target_name] = predsDataFrame
    print(predsDataFrame.head())

# Merge all targets in a single DataFrame
allpreds = pd.concat(
    [dataframe.assign(Target=target) for target, dataframe in predictions.items()],
    ignore_index=True
)


Predicting values for: CYC_Clontarf James Larkin Rd
251/251 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step
         date and time  Real  CNNLSTM_coords
0  2024-03-07 06:00:00    20              47
1  2024-03-07 07:00:00    49              50
2  2024-03-07 08:00:00    70              54
3  2024-03-07 09:00:00    51              57
4  2024-03-07 10:00:00    31              59

Predicting values for: CYC_Clontarf Pebble Beach Carpark
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step
         date and time  Real  CNNLSTM_coords
0  2024-03-07 06:00:00    26              18
1  2024-03-07 07:00:00    75              37
2  2024-03-07 08:00:00   105              55
3  2024-03-07 09:00:00    75              65
4  2024-03-07 10:00:00    33              64

Predicting values for: CYC_Grove Road Totem
251/251 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step
         date and time  Real  CNNLSTM_coords
0  2024-03-07 06:00:00   125             124
1  2024-03-07 07:00:00   309             188
2  2024-03-07 08:00:00   607             22

In [ ]:
allpreds.head(20)

,date and time,Real,CNNLSTM_coords,Target
0,2024-03-07 06:00:00,20,47,CYC_Clontarf James Larkin Rd
1,2024-03-07 07:00:00,49,50,CYC_Clontarf James Larkin Rd
2,2024-03-07 08:00:00,70,54,CYC_Clontarf James Larkin Rd
3,2024-03-07 09:00:00,51,57,CYC_Clontarf James Larkin Rd
4,2024-03-07 10:00:00,31,59,CYC_Clontarf James Larkin Rd
5,2024-03-07 11:00:00,32,58,CYC_Clontarf James Larkin Rd
6,2024-03-07 12:00:00,36,57,CYC_Clontarf James Larkin Rd
7,2024-03-07 13:00:00,33,57,CYC_Clontarf James Larkin Rd
8,2024-03-07 14:00:00,23,58,CYC_Clontarf James Larkin Rd
9,2024-03-07 15:00:00,25,61,CYC_Clontarf James Larkin Rd


## **4. RESULTS TO AZURE BLOB STORAGE**

In [ ]:
def save_blob(data, filename):
    try:
        blob_name = f"{CONTAINER_NAME}/{filename}"
        csv_data = data.to_csv(index=False)
        with fs.open(blob_name, "w") as f:
            f.write(csv_data)
        print(f"Saved to {blob_name}")
    except Exception as e:
        print(f"Error saving data to blob storage: {str(e)}")
        return False

In [ ]:
# Results
CONTAINER_NAME = "results"

save_blob(results, "rcnnlstm_coords.csv")

Saved to results/rcnnlstm_coords.csv


In [ ]:
# Predictions
CONTAINER_NAME = "predictions"

save_blob(allpreds, "pred_cnnlstm_coords.csv")

Saved to predictions/pred_cnnlstm_coords.csv


In [ ]:
allpreds.shape

(152190, 4)